In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Projects\ResearchPilot")

folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "src",
    "evaluation",
    "artifacts",
]

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Project folders created:")
for folder in folders:
    print(" -", folder)

print("\nProject root:", PROJECT_ROOT)

Project folders created:
 - data/raw
 - data/processed
 - notebooks
 - src
 - evaluation
 - artifacts

Project root: C:\Projects\ResearchPilot


In [2]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Projects\ResearchPilot")
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "papers"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PDF directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

PDF directory: C:\Projects\ResearchPilot\data\raw\papers
Processed data directory: C:\Projects\ResearchPilot\data\processed


In [3]:
import json
import time
import requests
import xml.etree.ElementTree as ET
from pathlib import Path

PAPER_IDS = [
    "1706.03762",  # Attention Is All You Need
    "2005.11401",  # Retrieval-Augmented Generation
    "1810.04805",  # BERT
    "1908.10084",  # Sentence-BERT
    "2004.04906",  # Dense Passage Retrieval
    "1702.08734",  # FAISS / GPU similarity search
]

API_URL = "https://export.arxiv.org/api/query"
NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom",
}

response = requests.get(
    API_URL,
    params={"id_list": ",".join(PAPER_IDS)},
    headers={"User-Agent": "ResearchPilot/1.0 (research project)"},
    timeout=60,
)
response.raise_for_status()

root = ET.fromstring(response.content)
papers = []

for entry in root.findall("atom:entry", NS):
    abs_url = entry.findtext("atom:id", default="", namespaces=NS)
    arxiv_id = abs_url.rstrip("/").split("/")[-1].split("v")[0]

    authors = [
        node.findtext("atom:name", default="", namespaces=NS)
        for node in entry.findall("atom:author", NS)
    ]

    papers.append({
        "arxiv_id": arxiv_id,
        "title": " ".join(
            entry.findtext("atom:title", default="", namespaces=NS).split()
        ),
        "authors": authors,
        "published": entry.findtext(
            "atom:published", default="", namespaces=NS
        ),
        "abstract": " ".join(
            entry.findtext("atom:summary", default="", namespaces=NS).split()
        ),
        "source_url": f"https://arxiv.org/abs/{arxiv_id}",
        "pdf_url": f"https://arxiv.org/pdf/{arxiv_id}",
    })

found_ids = {paper["arxiv_id"] for paper in papers}
missing_ids = set(PAPER_IDS) - found_ids

if missing_ids:
    raise RuntimeError(f"arXiv API did not return these IDs: {missing_ids}")

print(f"Retrieved metadata for {len(papers)} real papers.")
for paper in papers:
    print(f"\n{paper['arxiv_id']} — {paper['title']}")
    print("Authors:", ", ".join(paper["authors"][:4]))

Retrieved metadata for 6 real papers.

1706.03762 — Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit

1810.04805 — BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
Authors: Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova

1702.08734 — Billion-scale similarity search with GPUs
Authors: Jeff Johnson, Matthijs Douze, Hervé Jégou

2004.04906 — Dense Passage Retrieval for Open-Domain Question Answering
Authors: Vladimir Karpukhin, Barlas Oğuz, Sewon Min, Patrick Lewis

1908.10084 — Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks
Authors: Nils Reimers, Iryna Gurevych

2005.11401 — Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
Authors: Patrick Lewis, Ethan Perez, Aleksandra Piktus, Fabio Petroni


In [4]:
for paper in papers:
    pdf_path = RAW_DIR / f"{paper['arxiv_id']}.pdf"

    if pdf_path.exists() and pdf_path.stat().st_size > 10_000:
        print("Already downloaded:", pdf_path.name)
    else:
        pdf_response = requests.get(
            paper["pdf_url"],
            headers={"User-Agent": "ResearchPilot/1.0"},
            timeout=120,
        )
        pdf_response.raise_for_status()

        if not pdf_response.content.startswith(b"%PDF"):
            raise ValueError(
                f"Downloaded file is not a valid PDF: {paper['arxiv_id']}"
            )

        pdf_path.write_bytes(pdf_response.content)
        print("Downloaded:", pdf_path.name)

    paper["pdf_path"] = str(pdf_path)

    # Be considerate when making repeated requests.
    time.sleep(3)

metadata_path = PROCESSED_DIR / "paper_metadata.json"
metadata_path.write_text(
    json.dumps(papers, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"\nSaved metadata to: {metadata_path}")
print(f"PDF collection: {RAW_DIR}")

Already downloaded: 1706.03762.pdf
Already downloaded: 1810.04805.pdf
Already downloaded: 1702.08734.pdf
Already downloaded: 2004.04906.pdf
Already downloaded: 1908.10084.pdf
Already downloaded: 2005.11401.pdf

Saved metadata to: C:\Projects\ResearchPilot\data\processed\paper_metadata.json
PDF collection: C:\Projects\ResearchPilot\data\raw\papers


In [5]:
from pypdf import PdfReader

for paper in papers:
    reader = PdfReader(paper["pdf_path"])
    page_count = len(reader.pages)

    sample_text = "\n".join(
        (page.extract_text() or "")
        for page in reader.pages[:3]
    )

    paper["page_count"] = page_count
    paper["extracted_sample_chars"] = len(sample_text)

    print(
        f"{paper['arxiv_id']} | "
        f"{page_count} pages | "
        f"{len(sample_text):,} characters in first 3 pages"
    )

1706.03762 | 15 pages | 8,944 characters in first 3 pages
1810.04805 | 16 pages | 12,185 characters in first 3 pages
1702.08734 | 12 pages | 17,439 characters in first 3 pages
2004.04906 | 13 pages | 13,464 characters in first 3 pages
1908.10084 | 11 pages | 11,637 characters in first 3 pages
2005.11401 | 19 pages | 11,100 characters in first 3 pages
